# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Human Mast Cell Proteomics dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

> **Tip:** All entities (record sets, fields, columns, etc.) are referenced by their `@id` fields in this notebook for clarity and reproducibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs from the Croissant metadata.

Below, we print a summary of each record set (using its `@id`) and its available fields, so you can choose which parts to analyze further.

In [ ]:
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '')}")
        print(f"  Description: {rs.get('description', '')}")
        print("  Fields and columns:")
        for field in rs.get('field', []):
            print(f"    Field @id: {field['@id']} | Name: {field.get('name', '')} | DataType: {field.get('dataType', '')}")
            if 'column' in field:
                for col in field['column']:
                    print(f"      Column @id: {col['@id']} | Name: {col.get('name', '')}")
        print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis. **Remember:** You should use the record set `@id` and field `@id`s found in the previous overview.

Below, we will programmatically load all record sets found in the schema and print their columns. If the schema does not define any, you may need to manually specify a record set ID.

In [ ]:
# If schema contains no record sets, you may need to specify a plausible @id based on documentation or data download URLs.
record_set_ids = [rs['@id'] for rs in metadata.record_sets] if metadata.record_sets else []

dataframes = {}

if not record_set_ids:
    # Manually supply (from Croissant's distribution block):
    print("No record sets found in metadata. Attempting to extract records from available DataDownload distribution.")
    # Example: Find available data files
    dist = getattr(metadata, 'distribution', [])
    if not dist:
        print("No distributions found. Unable to load records.")
    else:
        # Since schema defines no explicit record sets, mlcroissant exposes a default record set per file
        # We'll get records from the first data file object.
        # You may replace this @id with the one you wish to load, if known.
        # The Croissant schema's main file appears to be:
        primary_distribution_id = dist[0]['@id']
        print(f"Reading records from distribution @id: {primary_distribution_id}")
        # This works if the file contains tabular records
        df = pd.DataFrame(list(dataset.records(record_set=primary_distribution_id)))
        dataframes[primary_distribution_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
        # For code demonstration, set the used record set ID
        example_record_set_id = primary_distribution_id
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet: {record_set_id}\nColumns: {df.columns.tolist()}\n")
        print(df.head(), "\n")
    example_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
 - Filter records based on numeric criteria.
 - Normalize numeric fields.
 - Optionally, group data (by a relevant categorical column).

> **Note:** For demonstration, you may replace the selected column names with ones suitable for the actual schema/data.

In [ ]:
# If no dataframes loaded above, nothing to analyze
if not dataframes:
    print("No dataframes available. Please ensure data is loaded in the previous step.")
else:
    # Select the dataframe and try to pick a numeric field for analysis
    record_set_id = example_record_set_id
    df = dataframes[record_set_id]

    print(f"Analyzing record set: {record_set_id}\nAvailable columns: {df.columns.tolist()}")

    # Attempt to auto-select a numeric field (float/int)
    numeric_cols = df.select_dtypes(include=["float", "int"]).columns.tolist()
    if not numeric_cols:
        print("No numeric columns detected for EDA. Please check the dataset.")
    else:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field (by @id or name): {numeric_field}")
        # Filter: values greater than a threshold (here, first quartile as example)
        threshold = df[numeric_field].quantile(0.25) if df[numeric_field].notna().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Optionally group by first string/object column
        group_fields = df.select_dtypes(include=["object", "category"]).columns.tolist()
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame().reset_index()
            print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The example below creates a histogram of the selected numeric field, and a boxplot grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not numeric_cols:
    print("No data available for visualization.")
else:
    # Histogram of numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze the FAIR² Human Mast Cell Proteomics dataset using the `mlcroissant` library. We examined schema record sets and fields using their `@id`s, loaded tabular records when available, conducted simple exploratory analyses, and visualized data distributions.

For deeper insights, consider referencing specific `@id` fields (see Section 2) and customizing the preprocessing or visualization code to your downstream workflow needs.